In [1]:
from tqdm import tqdm
import pandas as pd
import itertools
import random
import glob
import os
from pathlib import Path
import sys
BASE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from common.constants import DATA_DIR, OUTPUT_DIR

## Load wikidata aliases

In [2]:
path = os.path.join(DATA_DIR, "wikidata/processed/", "wb_items_per_site_filtered.csv")
df_filtered = pd.read_csv(path)

In [3]:
df_filtered.head()

,QID,Language,Title,CUI
0,Q3643466,af,Barnard se rower,C1681331
1,Q3642377,af,Rivier-dikkop,C3130926
2,Q3485242,af,Stegostomatidae,C1930548
3,Q567749,af,Xiphiidae,C0998964
4,Q2275677,als,Sexuelle Appetenzstörungen,C0036902


# Convert to SapBERT format

In [5]:
cleaned = (df_filtered[["CUI", "Title"]]
           .dropna(subset=["CUI", "Title"])
           .astype(str)
           .agg("||".join, axis=1)
           .tolist())


In [6]:
cleaned[0]

'C1681331||Barnard se rower'

### remove duplicates

In [7]:
print(len(cleaned))
cleaned = list(set(cleaned)) 
print(len(cleaned))

3834319
1737055


In [8]:
cleaned[:10]

['C0036941||Ајкула',
 'C0033166||Dilema do prisioneiro',
 'C2654002||Category:Penthetria heteroptera',
 'C1208950||흰얼굴기러기',
 'C0002810||Művészeti anatómia',
 'C0997173||Oryza officinalis',
 'C3576339||Kashmir nutcracker',
 'C0008641||Bakterienchromosom',
 'C2623508||Lophoglossus haldemanni',
 'C4110576||Helichrysum forsythii']

#  Optional: remove aliases in evaluation benchmarks
The released Wikidata alias resource is provided without benchmark-specific filtering. To reproduce the results reported in the paper, remove aliases that appear in the evaluation benchmarks using the code below.

In [ ]:
# xl-bel
xl_bel_path = os.path.join(DATA_DIR, "xl_bel/xlbel_v0.0/all_1k_test_query.txt")
with open(xl_bel_path, 'r') as f:
    xl_bel_queries = [line.strip() for line in f.readlines()]

In [ ]:
# EMEA
emea_path = glob.glob(os.path.join(DATA_DIR, "mantra/processed/EMEA", "*.txt"))
emea_queries = []
for path in emea_path:
    with open(path, 'r') as f:
        emea_queries.extend([line.strip() for line in f.readlines()])
emea_queries = [query.split("||")[:2] for query in emea_queries]
emea_queries = ["||".join(query) for query in emea_queries]

In [ ]:
# Patent
patent_path = os.path.join(DATA_DIR, "mantra/processed/Patent", "*.txt")
patent_queries = []
for path in glob.glob(patent_path):
    with open(path, 'r') as f:
        patent_queries.extend([line.strip() for line in f.readlines()])


patent_queries = [query.split("||")[:2] for query in patent_queries]
patent_queries = ["||".join(query) for query in patent_queries]

In [ ]:
# remove patent, emea, xl-bel queries from cleaned
cleaned_set = set(cleaned)
xl_bel_set = set(xl_bel_queries)
emea_set = set(emea_queries)
patent_set = set(patent_queries)
final_cleaned_set = cleaned_set - xl_bel_set - emea_set - patent_set
final_cleaned = list(final_cleaned_set)
print(f"Final cleaned size: {len(final_cleaned)}")

## positive pairs generation

### generate!

In [13]:
umls_dict = {} # constrauct cui to list of name dict
for line in tqdm(cleaned):
    cui, name = line.split("||")
    if cui in umls_dict:
        umls_dict[cui].append(name)
    else:
        umls_dict[cui] = [name]

100%|██████████| 1737055/1737055 [00:03<00:00, 462037.79it/s]


In [14]:
def gen_pairs(input_list):
    return list(itertools.combinations(input_list, r=2))

In [15]:
gen_pairs([1,2,3]) # test

[(1, 2), (1, 3), (2, 3)]

In [16]:
pos_pairs = []
for k,v in tqdm(umls_dict.items()):
    pairs = gen_pairs(v)
    #if len(pairs)>50: # if >50 pairs, then trim to 50 pairs
    #    pairs = random.sample(pairs, 50)
    for p in pairs:
        line = str(k) + "||" + p[0] + "||" + p[1]
        pos_pairs.append(line)

100%|██████████| 537848/537848 [00:11<00:00, 46039.38it/s] 


In [17]:
print(len(pos_pairs))

19094987


In [18]:
pos_pairs[:3]

['C0036941||Ајкула||منو',
 'C0036941||Ајкула||Kakya',
 'C0036941||Ајкула||Selachii']

### save the pairwise positive training file

In [19]:
debug_path = os.path.join(DATA_DIR, "sapbert", "debug_100.txt")
os.makedirs(os.path.dirname(debug_path), exist_ok=True)
with open(debug_path, 'w') as f:
    for line in pos_pairs[:100]:
        f.write("%s\n" % line)

In [21]:
with open(os.path.join(DATA_DIR, "sapbert", "training_file_wikidata2025_uncased_no_dup_pairwise_pair.txt"), 'w') as f:
    for line in pos_pairs:
        f.write("%s\n" % line)

In [22]:
pos_pairs = []
for k,v in tqdm(umls_dict.items()):
    pairs = gen_pairs(v)
    if len(pairs)>50: # if >50 pairs, then trim to 50 pairs
       pairs = random.sample(pairs, 50)
    for p in pairs:
        line = str(k) + "||" + p[0] + "||" + p[1]
        pos_pairs.append(line)
print(len(pos_pairs))

  0%|          | 0/537848 [00:00<?, ?it/s]

100%|██████████| 537848/537848 [00:04<00:00, 127715.02it/s]


2505046


In [24]:
with open(os.path.join(DATA_DIR, "sapbert", "training_file_wikidata2025_uncased_no_dup_pairwise_pair_50.txt"), 'w') as f:
    for line in pos_pairs:
        f.write("%s\n" % line)